# Understanding the Data
After unpacking the 2011-2023 Gwinnett zip file, I saw that there were actual a number of the excel spreadsheets which did contain sales information.

The format of this information is a little interesting, because there's several columns that are used to identify:
- LRSNum
- PIN
- LOCADDR
- LocCity
- LocZip
- LEGALAC
- PCDESC
- ZONEDESC

Then some attributes that are used to identify a specific owner, in this context a grantee.
- OWNER1
- OWNER2
- MAILADDR
- MAILCITY
- MAILSTAT
- Sale Date (Broken into SALE1D, SALE2D, SALE3D for the last transcations)
- Sale Amount (Broken into SALE1AMT, SALE2AMT, SALE3AMT)
- Grantor (Broken into GRANTOR1, GRANTOR2, GRANTOR3)
- Document Reference (Broken into DOC1REF, DOC2REF, DOC3REF) - I'm assuming something like the page and book numbers in other counties

The creation of the sales records is interesting because OWNER1 is the GRANTEE of GRANTOR1, GRANTOR1 is the GRANTEE of GRANTOR2, etc.  
So only some of the records will have OWNER1, OWNER2, MAILADDR, MAILCITY, MAILSTAT, specifically those records corresponding to the final sale of the contemporaneous owner (at time of recording).

**Road Map**  
1. Pick out a singular tax assessment document per year. There seems to be a lot of duplicate versions, with largely the same information. I'm just picking out the sheet that has the most rows / the latest sales date, indicating it's the most updated.
2. Check to ensure that the important columns are present across all of the assessment documents.
3. Duplicate the relevant property-wise columns for each of the sales transactions, to produce a unique row for each sale.
4. Merge all the years sales data.
5. Sort by those rows which have non-empty MAILADDR columns.
6. Deduplicate, taking the first value, ensuring that we take the most up to date sales transactions (those with MAILADDR), if possible

**Selected Tax Assessment Documents**  
Current Ownership_2011 Digest Assessed Values.xlsx  
2012 Gwinnett Digest TAFull_Ownership_CD7.xlsx  
2013 Tax Digest Ownership_CD7.xlsx  
2014 Property Ownership CD7.xlsx  
2015 Property Ownership CD7.xlsx  
2016 Property Ownership CD7.xlsx  
2017 Property Ownership CD7.xlsx  
2018 Property Ownership  CD7.xlsx  
2019 Property Ownership CD7.xlsx  
2020 Property Ownership CD7.xlsx  
2021 Property Ownership CD7.xlsx  
2022 Property Ownership CD7.xlsx  
2023 Property Ownership CD7.xlsx  

In [2]:
import pandas as pd
import os
from datetime import datetime

DATA_PATH = "../../data/gwinnett"
OUT_PATH = "../../data/gwinnett/out"

I have renamed all of hte files to follow the 20{XX} Property Ownership CD7.xlsx format, for convenience.

In [4]:
year_dfs = []

for file_p in os.listdir(DATA_PATH):
    if file_p.endswith(".xlsx"):
        print(file_p)
        year = int(file_p.split(" ")[0])
        df = pd.read_excel(os.path.join(DATA_PATH, file_p))

        year_dfs.append((year, df))

2023 Property Ownership CD7.xlsx
2019 Property Ownership CD7.xlsx
2016 Property Ownership CD7.xlsx
2013 Property Ownership CD7.xlsx
2018 Property Ownership CD7.xlsx
2022 Property Ownership CD7.xlsx
2012 Property Ownership CD7.xlsx
2017 Property Ownership CD7.xlsx
2014 Property Ownership CD7.xlsx
2011 Property Ownership CD7.xlsx
2021 Property Ownership CD7.xlsx
2015 Property Ownership CD7.xlsx
2020 Property Ownership CD7.xlsx


In [5]:
year_dfs.sort(key = lambda x : x[0])

In [6]:
# 2013 has an abnormal page structure
year_dfs[2] = (2013, pd.read_excel(os.path.join(DATA_PATH, "2013 Property Ownership CD7.xlsx"), sheet_name="real_master_0001"))

In [7]:
for year, df in year_dfs:
    print(year)
    print(df.columns)

2011
Index(['LRSNum', 'PIN', 'Public_NeiNum', 'LOCADDR', 'LocCity', 'LocState',
       'LocZip', 'OWNER1', 'OWNER2', 'MAILADDR', 'MAILCITY', 'MAILSTAT',
       'MAILZIP', 'LEGALAC', 'PCDESC', 'ZONEDESC', 'EXEMPT1', 'EXEMPT1D',
       'ASSMNT1D', 'LANDVAL1', 'DWLGVAL1', 'OTHVAL1', 'TOTVAL1', 'TAXLAND1',
       'TAXDWLG1', 'TAXOTH1', 'TAXTOT1', 'SALE1D', 'SALE2D', 'SALE3D',
       'SALE1AMT', 'SALE2AMT', 'SALE3AMT', 'GRANTOR1', 'GRANTOR2', 'GRANTOR3',
       'DOC1REF', 'DOC2REF', 'DOC3REF'],
      dtype='object')
2012
Index(['LRSNum', 'PIN', 'Public_NeiNum', 'LOCADDR', 'LocCity', 'LocState',
       'LocZip', 'OWNER1', 'OWNER2', 'MAILADDR', 'MAILCITY', 'MAILSTAT',
       'MAILZIP', 'LEGALAC', 'PCDESC', 'ZONEDESC', 'EXEMPT1', 'EXEMPT1D',
       'ASSMNT1D', 'LANDVAL1', 'DWLGVAL1', 'OTHVAL1', 'TOTVAL1', 'TAXLAND1',
       'TAXDWLG1', 'TAXOTH1', 'TAXTOT1', 'SALE1D', 'SALE2D', 'SALE3D',
       'SALE1AMT', 'SALE2AMT', 'SALE3AMT', 'GRANTOR1', 'GRANTOR2', 'GRANTOR3',
       'DOC1REF', 'DOC2REF', 

In [123]:
for year, df in year_dfs:
    print(f"Not in 2023, in {year}")
    print(set(df.columns).difference(set(year_dfs[-1][1].columns)))
    print(f"Not in {year}, in 2023")
    print(set(year_dfs[-1][1].columns).difference(set(df.columns)))
    print("\n")

Not in 2023, in 2011
{'LocCity', 'MAILCITY', 'GRANTOR1', 'GRANTOR2', 'LOCADDR', 'LocState', 'GRANTOR3', 'EXEMPT1D', 'OWNER1', 'EXEMPT1', 'OWNER2', 'MAILADDR', 'MAILSTAT', 'MAILZIP', 'LocZip'}
Not in 2011, in 2023
{'OWNERNAME1', 'GRANTORNAME2', 'PROPCLAS', 'DISTNUM_DESC', 'PROPERTYState', 'OWNERCITY', 'OWNERZIP', 'OWNERADDRESS1', 'LEGAL1', 'OWNERSTATE', 'DISTNUM', 'PROPCLAS_DESC', 'OWNERNAME2', 'GRANTORNAME3', 'PROPERTYZip', 'PROPERTYCity', 'PROPERTYSTREET', 'GRANTORNAME1'}


Not in 2023, in 2012
{'LocCity', 'MAILCITY', 'GRANTOR1', 'GRANTOR2', 'LOCADDR', 'LocState', 'GRANTOR3', 'EXEMPT1D', 'OWNER1', 'EXEMPT1', 'OWNER2', 'MAILADDR', 'MAILSTAT', 'MAILZIP', 'LocZip'}
Not in 2012, in 2023
{'OWNERNAME1', 'GRANTORNAME2', 'PROPCLAS', 'DISTNUM_DESC', 'PROPERTYState', 'OWNERCITY', 'OWNERZIP', 'OWNERADDRESS1', 'LEGAL1', 'OWNERSTATE', 'DISTNUM', 'PROPCLAS_DESC', 'OWNERNAME2', 'GRANTORNAME3', 'PROPERTYZip', 'PROPERTYCity', 'PROPERTYSTREET', 'GRANTORNAME1'}


Not in 2023, in 2013
{'LocCity', 'MAILCI

It looks like 2011 and 2012 have their own format, 2013 has its own format, and the rest of the years are the same. So I will be parsing them using the columns of the 2023 dataset.

In [8]:
property_cols = ['LRSNum', 'PIN', 'Public_NeiNum', 'LOCADDR', 'LocCity', 'LocState',
       'LocZip', 'LEGALAC', 'PCDESC', 'ZONEDESC']

owner_cols = ['OWNER1', 'OWNER2', 'MAILADDR', 'MAILCITY', 'MAILSTAT',
       'MAILZIP']

drop_cols = ['EXEMPT1', 'EXEMPT1D',
       'ASSMNT1D', 'LANDVAL1', 'DWLGVAL1', 'OTHVAL1', 'TOTVAL1', 'TAXLAND1',
       'TAXDWLG1', 'TAXOTH1', 'TAXTOT1']

sale_cols = ['SALE1D', 'SALE2D', 'SALE3D',
       'SALE1AMT', 'SALE2AMT', 'SALE3AMT', 'GRANTOR1', 'GRANTOR2', 'GRANTOR3',
       'DOC1REF', 'DOC2REF', 'DOC3REF']

sales_2011_2013 = []
for i in range(3):
    year, year_df = year_dfs[i]
    year_df['SALE1D'] = pd.to_datetime(year_df['SALE1D'], format="%m/%d/%Y", errors="coerce")
    year_df['SALE2D'] = pd.to_datetime(year_df['SALE2D'], format="%m/%d/%Y", errors="coerce")
    year_df['SALE3D'] = pd.to_datetime(year_df['SALE3D'], format="%m/%d/%Y", errors="coerce")
    year_df['YEAR_RECORDED'] = year
    grantor3_sales = year_df.loc[~year_df['SALE3D'].isna(), :].copy()
    grantor2_sales = year_df.loc[~year_df['SALE2D'].isna(), :].copy()
    grantor1_sales = year_df.loc[~year_df['SALE1D'].isna(), :].copy()
    
    grantor3_sales['GRANTOR'] = grantor3_sales['GRANTOR3']
    grantor3_sales['GRANTEE'] = grantor3_sales['GRANTOR2']
    grantor3_sales['SALEDT'] = grantor3_sales['SALE3D']
    grantor3_sales['SALEAMT'] = grantor3_sales['SALE3AMT']
    grantor3_sales['DOCREF'] = grantor3_sales['DOC3REF']
    grantor3_sales.loc[:, owner_cols] = pd.NA
    grantor3_sales = grantor3_sales.drop(columns=drop_cols + sale_cols)

    grantor2_sales['GRANTOR'] = grantor2_sales['GRANTOR2']
    grantor2_sales['GRANTEE'] = grantor2_sales['GRANTOR1']
    grantor2_sales['SALEDT'] = grantor2_sales['SALE2D']
    grantor2_sales['SALEAMT'] = grantor2_sales['SALE2AMT']
    grantor2_sales['DOCREF'] = grantor2_sales['DOC2REF']
    grantor2_sales.loc[:, owner_cols] = pd.NA
    grantor2_sales = grantor2_sales.drop(columns=drop_cols + sale_cols)

    grantor1_sales['GRANTOR'] = grantor1_sales['GRANTOR1']
    grantor1_sales['GRANTEE'] = grantor1_sales['OWNER1']
    grantor1_sales['SALEDT'] = grantor1_sales['SALE1D']
    grantor1_sales['SALEAMT'] = grantor1_sales['SALE1AMT']
    grantor1_sales['DOCREF'] = grantor1_sales['DOC1REF']
    # Notably, no clearing of the owner1 columns this time because they actually exist
    grantor1_sales = grantor1_sales.drop(columns=drop_cols + sale_cols)

    sales_2011_2013.append(grantor1_sales)
    sales_2011_2013.append(grantor2_sales)
    sales_2011_2013.append(grantor3_sales)

sales_2011_2013_df = pd.concat(sales_2011_2013)

In [9]:
# Normalize with the other dataframes
sales_2011_2013_df = sales_2011_2013_df.rename(columns={'LOCADDR': 'PROPERTYSTREET', 'LocCity': 'PROPERTYCity',
                                                        'LocState': 'PROPERTYState', 'LocZip': 'PROPERTYZip',
                                                        'OWNER1': 'OWNERNAME1', 'OWNER2': 'OWNERNAME2',
                                                        'MAILADDR': 'OWNERADDRESS1', 'MAILCITY': "OWNERCITY",
                                                        "MAILSTAT": "OWNERSTATE", "MAILZIP": "OWNERZIP"})

In [10]:
property_cols = ['LRSNum', 'PIN', 'Public_NeiNum', 'PROPERTYSTREET', 'PROPERTYCity',
       'PROPERTYState', 'PROPERTYZip', 'LEGALAC', 'LEGAL1', 'PCDESC', 'ZONEDESC']

owner_cols = ['OWNERNAME1', 'OWNERNAME2',
       'OWNERADDRESS1', 'OWNERCITY', 'OWNERSTATE', 'OWNERZIP']

drop_cols = ['ASSMNT1D', 'LANDVAL1', 'DWLGVAL1', 'OTHVAL1', 'TOTVAL1', 'TAXLAND1', 'TAXDWLG1',
       'TAXOTH1', 'TAXTOT1', 'LEGAL1', ]

sale_cols = ['SALE1D', 'SALE2D', 'SALE3D', 'SALE1AMT',
       'SALE2AMT', 'SALE3AMT', 'GRANTORNAME1', 'GRANTORNAME2', 'GRANTORNAME3',
       'DOC1REF', 'DOC2REF', 'DOC3REF']

sales_2014_2023 = []
for i in range(3, len(year_dfs)):
    year, year_df = year_dfs[i]
    year_df['SALE1D'] = pd.to_datetime(year_df['SALE1D'], format="%m/%d/%Y", errors="coerce")
    year_df['SALE2D'] = pd.to_datetime(year_df['SALE2D'], format="%m/%d/%Y", errors="coerce")
    year_df['SALE3D'] = pd.to_datetime(year_df['SALE3D'], format="%m/%d/%Y", errors="coerce")
    year_df['YEAR_RECORDED'] = year
    grantor3_sales = year_df.loc[~year_df['SALE3D'].isna(), :].copy()
    grantor2_sales = year_df.loc[~year_df['SALE2D'].isna(), :].copy()
    grantor1_sales = year_df.loc[~year_df['SALE1D'].isna(), :].copy()
    
    grantor3_sales['GRANTOR'] = grantor3_sales['GRANTORNAME3']
    grantor3_sales['GRANTEE'] = grantor3_sales['GRANTORNAME2']
    grantor3_sales['SALEDT'] = grantor3_sales['SALE3D']
    grantor3_sales['SALEAMT'] = grantor3_sales['SALE3AMT']
    grantor3_sales['DOCREF'] = grantor3_sales['DOC3REF']
    grantor3_sales.loc[:, owner_cols] = pd.NA
    grantor3_sales = grantor3_sales.drop(columns=drop_cols + sale_cols)

    grantor2_sales['GRANTOR'] = grantor2_sales['GRANTORNAME2']
    grantor2_sales['GRANTEE'] = grantor2_sales['GRANTORNAME1']
    grantor2_sales['SALEDT'] = grantor2_sales['SALE2D']
    grantor2_sales['SALEAMT'] = grantor2_sales['SALE2AMT']
    grantor2_sales['DOCREF'] = grantor2_sales['DOC2REF']
    grantor2_sales.loc[:, owner_cols] = pd.NA
    grantor2_sales = grantor2_sales.drop(columns=drop_cols + sale_cols)

    grantor1_sales['GRANTOR'] = grantor1_sales['GRANTORNAME1']
    grantor1_sales['GRANTEE'] = grantor1_sales['OWNERNAME1']
    grantor1_sales['SALEDT'] = grantor1_sales['SALE1D']
    grantor1_sales['SALEAMT'] = grantor1_sales['SALE1AMT']
    grantor1_sales['DOCREF'] = grantor1_sales['DOC1REF']
    # Notably, no clearing of the owner1 columns this time because they actually exist
    grantor1_sales = grantor1_sales.drop(columns=drop_cols + sale_cols)

    sales_2014_2023.append(grantor1_sales)
    sales_2014_2023.append(grantor2_sales)
    sales_2014_2023.append(grantor3_sales)

sales_2014_2023_df = pd.concat(sales_2014_2023)
sales_2014_2023_df = sales_2014_2023_df.drop(columns=['EXEMPT1', 'EXEMPT1D'])

In [11]:
total_sales_digest = pd.concat([sales_2011_2013_df, sales_2014_2023_df])

In [12]:
total_sales_digest['HASOWNER'] = ~total_sales_digest['OWNERNAME1'].isna()
total_sales_digest['PIN'] = total_sales_digest['PIN'].str.strip()

In [22]:
total_sales_digest['SALE_YR'] = total_sales_digest['SALEDT'].dt.year
total_sales_digest.loc[total_sales_digest['SALE_YR'] > (total_sales_digest['YEAR_RECORDED'] + 1), "SALEDT"] -= pd.offsets.DateOffset(years=100)
total_sales_digest['SALE_YR'] = total_sales_digest['SALEDT'].dt.year

In [24]:
total_sales_digest['SALE_YR'].value_counts().sort_index(ascending=False)

SALE_YR
2023     11172
2022     59765
2021    115275
2020    125911
2019    157545
         ...  
1909         3
1907        13
1906        13
1905        32
1904        39
Name: count, Length: 79, dtype: int64

In [25]:
total_sales_digest = total_sales_digest.sort_values(by="YEAR_RECORDED", ascending=True)
total_sales_digest = total_sales_digest.sort_values(by="HASOWNER", ascending=False)

I can't actually distinguish for sure which one is the "ParcelID" equivalent for Gwinnett county. Both LRSNum and PIN seem to be unique. I just choose PIN

In [26]:
original_rows = total_sales_digest.shape[0]
total_sales_digest_dedup = total_sales_digest.drop_duplicates(subset=['PIN', 'SALEDT'])
new_rows = total_sales_digest_dedup.shape[0]

print(f"Rows Removed: {original_rows - new_rows}, {new_rows} Remaining")

Rows Removed: 8184756, 937362 Remaining


This passes the sniff test, since the total number of 2023 rows was around 300000, so assuming that there were some properties which were sold more than 3 times, wherein the 2012 file contributed different sales records than 2023 for example, then this checks out.

In [27]:
total_sales_digest_dedup = total_sales_digest_dedup.sort_values(by="SALEDT", ascending=True)

In [ ]:
total_sales_digest_dedup.to_csv(os.path.join(OUT_PATH, "GWINNET_SALES_FINAL.csv"), index=False)

# Merging with Tax Digest
The sales digest in this scenario is actually derived from the tax digest, so it is a little redundant, but I guess it's a reverse mapping if you need that.

In [ ]:
TAX_DIGEST_PATH = "/Users/tpeng/Library/CloudStorage/OneDrive-GeorgiaInstituteofTechnology/Housing and Urban Policy (HUP) Lab - Documents/Data Files/Final Datasets/Gwinnett/gwinnett_digest_withnonprofit.csv"
SALES_DIGEST_PATH = os.path.join(OUT_PATH, "GWINNET_SALES_FINAL.csv")

print(f"Correct Tax Digest Path {os.path.exists(TAX_DIGEST_PATH)}")
print(f"Correct Sales Digest Path {os.path.exists(SALES_DIGEST_PATH)}")

Correct Tax Digest Path True
Correct Sales Digest Path True


In [34]:
tax_digest = pd.read_csv(TAX_DIGEST_PATH, dtype={"property_zip": object, "legal1": object, "distnum": object, 
                                                 "distnum_desc": object, "propclas_desc": object,
                                                 "mod_ownerzip": object}, 
                                                 parse_dates=["assmnt1d", "sale1d", "sale2d", "sale3d"])

In [44]:
tax_digest.columns

Index(['tax_year', 'lrs_num', 'pin', 'public_nei_num', 'propertystreet',
       'property_city', 'property_state', 'property_zip', 'ownername1',
       'ownername2', 'owneraddress1', 'ownercity', 'ownerstate', 'ownerzip',
       'legalac', 'pcdesc', 'zonedesc', 'exempt1', 'exempt1d', 'assmnt1d',
       'landval1', 'dwlgval1', 'othval1', 'totval1', 'taxland1', 'taxdwlg1',
       'taxoth1', 'taxtot1', 'sale1d', 'sale2d', 'sale3d', 'sale1amt',
       'sale2amt', 'sale3amt', 'grantorname1', 'grantorname2', 'grantorname3',
       'doc1ref', 'doc2ref', 'doc3ref', 'legal1', 'propclas', 'distnum',
       'distnum_desc', 'propclas_desc', 'mod_own_adrstr', 'mod_unitno',
       'mod_ownerzip', 'owner_addr', 'own_corp_flag', 'rental_flag',
       'mod_own_adrsuf2', 'mod_own_adrsuf', 'mod_owneraddress1',
       'mod_owneraddress1_B', 'TAXPIN', 'longitude', 'latitude', 'owner_type'],
      dtype='object')

There are duplicate year - pin, year - lrs_num pairs in the gwinnett tax digest.

In [38]:
# Determine which of LRSNum and Pin are the unique identifier
print(f"lrs_num unique: {~tax_digest.duplicated(subset=['lrs_num', 'tax_year']).any()}")
conflicts = tax_digest[tax_digest.duplicated(subset=['lrs_num', 'tax_year'], keep=False)]
print(conflicts)
print("\n")

print(f"pin unique: {~tax_digest.duplicated(subset=['pin', 'tax_year']).any()}")
conflicts = tax_digest[tax_digest.duplicated(subset=['pin', 'tax_year'], keep=False)]
print(conflicts)
print("\n")

lrs_num unique: False
         tax_year  lrs_num        pin  public_nei_num         propertystreet  \
2378         2011   237396  R1003 123            8800  3531 THOMPSON MILL RD   
2379         2011   237396  R1003 123            8800  3531 THOMPSON MILL RD   
2801         2011   238201  R1004 014            9035  3774 THOMPSON MILL RD   
2802         2011   238201  R1004 014            9035  3774 THOMPSON MILL RD   
3828         2011   239968  R2001 091            2013    650 BAILEY WOODS RD   
...           ...      ...        ...             ...                    ...   
3691549      2023  3337212  R7312 136            7388           BRENDLYNN CT   
3691550      2023  3337212  R7312 136            7388           BRENDLYNN CT   
3691551      2023  3337212  R7312 136            7388           BRENDLYNN CT   
3691552      2023  3337212  R7312 136            7388           BRENDLYNN CT   
3691553      2023  3337212  R7312 136            7388           BRENDLYNN CT   

        property_

In [48]:
conflicts.to_csv(os.path.join(OUT_PATH, "GWINNETT_DIGEST_PIN_YEAR_DUPES.csv"))

In [49]:
digest_duplicates = tax_digest[tax_digest.duplicated()]
digest_duplicates.to_csv(os.path.join(OUT_PATH, "GWINNETT_FULL_DIGEST_DUPES.csv"))

In [43]:
sales_digest = pd.read_csv(SALES_DIGEST_PATH, low_memory=False, parse_dates=["SALEDT"])

In [46]:
merge_cols = ['tax_year', 'lrs_num', 'pin', 'public_nei_num', 'propertystreet',
       'property_city', 'property_state', 'property_zip', 'ownername1',
       'ownername2', 'owneraddress1', 'ownercity', 'ownerstate', 'ownerzip',
       'legalac', 'pcdesc', 'zonedesc', 'exempt1', 'exempt1d', 'assmnt1d',
       'landval1', 'dwlgval1', 'othval1', 'totval1', 'taxland1', 'taxdwlg1',
       'taxoth1', 'taxtot1', 'propclas', 'distnum', 'distnum_desc', 'propclas_desc',
       'mod_own_adrstr', 'mod_unitno', 'mod_ownerzip', 'owner_addr', 'own_corp_flag',
       'rental_flag', 'mod_own_adrsuf2', 'mod_own_adrsuf', 'mod_owneraddress1',
       'mod_owneraddress1_B', 'TAXPIN', 'longitude', 'latitude', 'owner_type']

tax_subset = tax_digest[merge_cols]
sales_subset = sales_digest[(sales_digest['SALE_YR'] >= 2011) & (sales_digest['SALE_YR'] <= 2023)]
sales_tax_merged = pd.merge(sales_subset, tax_subset, how="left", left_on=["PIN", "SALE_YR"], right_on=["pin", "tax_year"], suffixes=("_SALES", "_DIGEST"))

total_rows = sales_subset.shape[0]
merged_rows = sales_tax_merged['pin'].notna().sum()

print(f"{merged_rows} out of {total_rows} matched: {merged_rows / total_rows * 100}%")

391313 out of 399249 matched: 98.0122680332324%


In [50]:
sales_tax_merged.to_csv(os.path.join(OUT_PATH, "GWINNET_SALES_TAX_DIGEST_FINAL.csv"), index=False)

# Diagnosing Unmatched

In [53]:
unmerged_df = sales_tax_merged[sales_tax_merged["pin"].isna()]

In [57]:
unmerged_pids = pd.Series(unmerged_df["PIN"].unique())
print(f"{unmerged_pids.size} unmerged unique parcelIDs")

7090 unmerged unique parcelIDs


In [58]:
not_present = unmerged_pids[~unmerged_pids.isin(tax_digest["pin"])]
present = unmerged_pids[unmerged_pids.isin(tax_digest["pin"])]
print(f"{not_present.size} unmerged parcelIDs not present in digest")

0 unmerged parcelIDs not present in digest


In [59]:
unmerged_not_present_df = unmerged_df[unmerged_df["PIN"].isin(not_present)]
unmerged_present_df = unmerged_df[unmerged_df["PIN"].isin(present)]

In [65]:
unmerged_present_df["year_before"] = unmerged_present_df["SALE_YR"] - 1
unmerged_present_df["year_after"] = unmerged_present_df["SALE_YR"] + 1

unmerged_year_status_df = pd.merge(unmerged_present_df, tax_digest[["pin", "tax_year"]], left_on=["PIN", "year_before"], right_on=["pin", "tax_year"], how="left", suffixes=("", "_before"))
unmerged_year_status_df = pd.merge(unmerged_year_status_df, tax_digest[["pin", "tax_year"]], left_on=["PIN", "year_after"], right_on=["pin", "tax_year"], how="left", suffixes=("", "_after"))

In [66]:
before = (~unmerged_year_status_df.pin_before.isna()).sum()
after = (~unmerged_year_status_df.pin_after.isna()).sum()
both = ((~unmerged_year_status_df.pin_before.isna()) & (~unmerged_year_status_df.pin_after.isna())).sum()

print(f"{before}/{unmerged_present_df.shape[0]} parcels existed before sale\n{after}/{unmerged_present_df.shape[0]} parcels existed after \n{both}/{unmerged_present_df.shape[0]} parcels existed both before and after sale.\n" + 
      f"{unmerged_present_df.shape[0] - max(before, after, both)}/{unmerged_present_df.shape[0]} parcels existed neither before nor after.")

24/8108 parcels existed before sale
8043/8108 parcels existed after 
13/8108 parcels existed both before and after sale.
65/8108 parcels existed neither before nor after.


In [68]:
unmerged_present_df.to_csv(os.path.join(OUT_PATH, "GWINNETT_UNMERGED_PRESENT.csv"), index=False)